# Keyword Append

Appends results from multiple search keywords into one dataset with keyword priority rules and de-duplication by seller item ID.

**Setup.** Data paths point to `/content/drive/MyDrive/YOUR_PROJECT_FOLDER/...` — change `YOUR_PROJECT_FOLDER` to your own folder before running. API keys are read from Colab Secrets / environment variables, never hard-coded.

> Code comments and console messages are in Korean (original working language). See the [folder README](README.md) for the pipeline overview and the [disclaimer](../README.md#disclaimer).


In [ ]:
# ================================================================================
# 1. Google Drive 연결
# ================================================================================
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ================================================================================
# 2. Google Drive 폴더 설정
# ================================================================================

from pathlib import Path

INPUT_DIR = Path("/content/drive/MyDrive/YOUR_PROJECT_FOLDER/data_coupang")

print("폴더 존재 여부 :", INPUT_DIR.exists())
print("폴더 경로 :", INPUT_DIR)

xlsx_files = [
    f for f in INPUT_DIR.glob("*.xlsx")
    if not f.name.startswith("~$")
]

print()
print("=" * 80)
print("폴더 내 엑셀 파일")
print("=" * 80)

for f in xlsx_files:
    print(f.name)

print()
print("총 파일 수 :", len(xlsx_files))

INPUT_DIR = Path("/content/drive/MyDrive/YOUR_PROJECT_FOLDER/data_coupang")

OUTPUT_FILE = INPUT_DIR / "쿠팡_전체_제품분석_통합.xlsx"

In [ ]:
# ================================================================================
# 쿠팡 제품분석 + 가격요약 시트 통합
# Google Drive > 260824-260921 > coupang_pricedata
#
# 통합 대상 시트
#   1. 제품분석
#   2. 가격요약
#
# APPEND 우선순위
#   주름패치 → 피부패치 → 패치
#   성인용기저귀 → 기저귀
#   입는생리대 → 생리대
#   물티슈
#
# 같은 itemId 또는 vendorItemId가 여러 키워드에 존재하면
# 먼저 APPEND된 키워드의 데이터를 남김
# ================================================================================

from pathlib import Path
import pandas as pd
import re
from google.colab import drive, files

# ================================================================================
# 3. APPEND 우선순위
# ================================================================================
#
# 세부 키워드 → 상위 키워드 순서
#
# 주름패치 상품이 패치에도 있으면
# → 주름패치 데이터를 먼저 보존
#
# 성인용기저귀 상품이 기저귀에도 있으면
# → 성인용기저귀 데이터를 먼저 보존
#
# 입는생리대 상품이 생리대에도 있으면
# → 입는생리대 데이터를 먼저 보존
# ================================================================================

APPEND_ORDER = [
    "주름패치",
    "피부패치",
    "패치",

    "성인용기저귀",
    "기저귀",

    "입는생리대",
    "생리대",

    "물티슈",
]

ORDER_MAP = {
    keyword: i
    for i, keyword in enumerate(APPEND_ORDER)
}


# ================================================================================
# 4. 폴더 확인
# ================================================================================

print()
print("=" * 80)
print("① 통합 대상 폴더")
print("=" * 80)

print("폴더 :", INPUT_DIR)

if not INPUT_DIR.exists():
    raise FileNotFoundError(
        f"""
Google Drive 폴더를 찾을 수 없습니다.

확인할 경로:
{INPUT_DIR}
"""
    )


# ================================================================================
# 5. 엑셀 파일 확인
# ================================================================================

xlsx_files = [
    f
    for f in INPUT_DIR.glob("*.xlsx")
    if not f.name.startswith("~$")
    and f.name != OUTPUT_FILE.name
]

print("엑셀 파일 수 :", len(xlsx_files))

if len(xlsx_files) == 0:
    raise FileNotFoundError(
        f"""
해당 폴더에 통합할 xlsx 파일이 없습니다.

폴더:
{INPUT_DIR}
"""
    )


# ================================================================================
# 6. 각 파일의 검색키워드 확인
# ================================================================================

print()
print("=" * 80)
print("② 파일별 검색키워드 확인")
print("=" * 80)

file_info = []

for file in xlsx_files:

    try:

        temp_df = pd.read_excel(
            file,
            sheet_name="제품분석",
            usecols=["검색키워드"]
        )

        keywords = (
            temp_df["검색키워드"]
            .dropna()
            .astype(str)
            .str.strip()
        )

        keywords = keywords[
            keywords.ne("")
            & keywords.ne("검색어없음")
        ]

        if len(keywords) == 0:
            keyword = ""
        else:
            keyword = keywords.iloc[0]

        file_info.append({
            "file": file,
            "keyword": keyword
        })

        print(
            f"[확인] "
            f"{keyword:<10} "
            f"{file.name}"
        )

    except Exception as e:

        print()
        print(f"[키워드 확인 실패] {file.name}")
        print(f"오류 : {e}")


# ================================================================================
# 7. APPEND 순서대로 정렬
# ================================================================================

file_info = sorted(
    file_info,
    key=lambda x: ORDER_MAP.get(
        x["keyword"],
        9999
    )
)


print()
print("=" * 80)
print("③ 최종 APPEND 순서")
print("=" * 80)

for i, info in enumerate(file_info, start=1):

    print(
        f"{i:>2}. "
        f"{info['keyword']:<10} "
        f"{info['file'].name}"
    )


# ================================================================================
# 8. ID 정규화 함수
# ================================================================================

def normalize_id(value):

    if pd.isna(value):
        return ""

    value = str(value).strip()

    # Excel에서 숫자형 ID가 123456.0으로 읽힌 경우
    if re.fullmatch(r"\d+\.0", value):
        value = value[:-2]

    return value



In [ ]:
# ================================================================================
# 9. 특정 시트 통합 함수
# ================================================================================

def merge_sheet(
    sheet_name,
    file_info,
    dedup_mode="product",
    need_keyword=False
):

    print()
    print("=" * 80)
    print(f"{sheet_name} 시트 통합")
    print("=" * 80)

    dfs = []

    # =============================================================================
    # 9-A. 파일 순서대로 해당 시트 읽기
    # =============================================================================

    for info in file_info:

        file = info["file"]
        keyword = info["keyword"]

        try:

            df = pd.read_excel(
                file,
                sheet_name=sheet_name
            )

            # 컬럼명 정리
            df.columns = (
                df.columns
                .astype(str)
                .str.strip()
            )

            # 빈 행 제거
            df = df.dropna(
                axis=0,
                how="all"
            )

            # 빈 열 제거
            df = df.dropna(
                axis=1,
                how="all"
            )

            if len(df) == 0:
                continue

            # 검색키워드 컬럼이 없는 경우
            if need_keyword:

                if "검색키워드" not in df.columns:

                    df["검색키워드"] = keyword

                else:

                    df["검색키워드"] = (
                        df["검색키워드"]
                        .fillna("")
                        .astype(str)
                        .str.strip()
                    )

                    empty_mask = (
                        df["검색키워드"].eq("")
                        | df["검색키워드"].eq("검색어없음")
                    )

                    df.loc[
                        empty_mask,
                        "검색키워드"
                    ] = keyword

            # 원본 파일
            df["_원본파일"] = file.name

            # 검색키워드 APPEND 순서
            df["_append순서"] = ORDER_MAP.get(
                keyword,
                9999
            )

            dfs.append(df)

            print(
                f"[APPEND] "
                f"{keyword:<10} "
                f"{len(df):>6}행   "
                f"{file.name}"
            )

        except ValueError:

            print(
                f"[시트 없음] "
                f"{keyword:<10} "
                f"{file.name} → {sheet_name}"
            )

        except Exception as e:

            print(
                f"[실패] {file.name} → {sheet_name}"
            )
            print(e)


    if not dfs:
        raise ValueError(
            f"'{sheet_name}' 시트를 읽은 파일이 없습니다."
        )


    # =============================================================================
    # 9-B. 전체 APPEND
    # =============================================================================

    merged_df = pd.concat(
        dfs,
        ignore_index=True,
        sort=False
    )

    print()
    print(
        "통합 전 행 수 :",
        len(merged_df)
    )


    # =============================================================================
    # 9-C. ID 컬럼 준비
    # =============================================================================

    if "itemId" not in merged_df.columns:
        merged_df["itemId"] = ""

    if "vendorItemId" not in merged_df.columns:
        merged_df["vendorItemId"] = ""

    merged_df["_itemId_key"] = (
        merged_df["itemId"]
        .apply(normalize_id)
    )

    merged_df["_vendorItemId_key"] = (
        merged_df["vendorItemId"]
        .apply(normalize_id)
    )


    # =============================================================================
    # 9-D. 제품분석 / 가격요약
    # =============================================================================

    if dedup_mode == "product":

        seen_item_ids = set()
        seen_vendor_ids = set()

        keep_indices = []

        for idx, row in merged_df.iterrows():

            item_id = row["_itemId_key"]
            vendor_id = row["_vendorItemId_key"]

            item_duplicate = (
                item_id != ""
                and item_id in seen_item_ids
            )

            vendor_duplicate = (
                vendor_id != ""
                and vendor_id in seen_vendor_ids
            )

            if item_duplicate or vendor_duplicate:
                continue

            keep_indices.append(idx)

            if item_id:
                seen_item_ids.add(item_id)

            if vendor_id:
                seen_vendor_ids.add(vendor_id)

        result_df = merged_df.loc[
            keep_indices
        ].copy()


    # =============================================================================
    # 9-E. 가격이력
    # =============================================================================

    elif dedup_mode == "price_history":

        # 날짜 컬럼 확인
        if "날짜" not in merged_df.columns:

            raise ValueError(
                "가격이력 시트에 '날짜' 컬럼이 없습니다."
            )

        # 날짜 정규화
        merged_df["_날짜_key"] = pd.to_datetime(
            merged_df["날짜"],
            errors="coerce"
        )

        # 가격이력은
        #
        # 상품 ID + 날짜
        #
        # 를 기준으로 중복 제거한다.
        #
        # 따라서 같은 상품이라도 날짜가 다르면 모두 유지된다.

        seen_item_date = set()
        seen_vendor_date = set()

        keep_indices = []

        for idx, row in merged_df.iterrows():

            item_id = row["_itemId_key"]
            vendor_id = row["_vendorItemId_key"]
            date_value = row["_날짜_key"]

            # 날짜가 정상적으로 파싱되지 않은 경우
            # 원본 날짜 문자열도 보조키로 사용
            if pd.isna(date_value):

                date_key = str(
                    row["날짜"]
                ).strip()

            else:

                date_key = date_value

            item_date_key = None
            vendor_date_key = None

            if item_id:

                item_date_key = (
                    item_id,
                    date_key
                )

            if vendor_id:

                vendor_date_key = (
                    vendor_id,
                    date_key
                )

            item_duplicate = (
                item_date_key is not None
                and item_date_key in seen_item_date
            )

            vendor_duplicate = (
                vendor_date_key is not None
                and vendor_date_key in seen_vendor_date
            )

            # 같은 상품 + 같은 날짜/시점이면 제거
            if item_duplicate or vendor_duplicate:
                continue

            keep_indices.append(idx)

            if item_date_key is not None:
                seen_item_date.add(
                    item_date_key
                )

            if vendor_date_key is not None:
                seen_vendor_date.add(
                    vendor_date_key
                )

        result_df = merged_df.loc[
            keep_indices
        ].copy()

    else:

        raise ValueError(
            f"알 수 없는 dedup_mode: {dedup_mode}"
        )


    # =============================================================================
    # 9-F. 가격이력 날짜 정렬
    # =============================================================================

    if dedup_mode == "price_history":

        # 실제 날짜 기준으로 정렬
        result_df["_sort_date"] = pd.to_datetime(
            result_df["날짜"],
            errors="coerce"
        )

        result_df = result_df.sort_values(
            by=[
                "_append순서",
                "_sort_date"
            ],
            ascending=[
                True,
                True
            ],
            na_position="last"
        )

        result_df.drop(
            columns=[
                "_sort_date"
            ],
            errors="ignore",
            inplace=True
        )


    # =============================================================================
    # 9-G. 임시 컬럼 제거
    # =============================================================================

    result_df.drop(
        columns=[
            "_itemId_key",
            "_vendorItemId_key",
            "_날짜_key",
            "_append순서"
        ],
        errors="ignore",
        inplace=True
    )


    # =============================================================================
    # 9-H. 원본 파일은 마지막
    # =============================================================================

    if "_원본파일" in result_df.columns:

        original_columns = [
            c
            for c in result_df.columns
            if c != "_원본파일"
        ]

        result_df = result_df[
            original_columns + ["_원본파일"]
        ]


    print()
    print(
        "통합 후 행 수 :",
        len(result_df)
    )

    print(
        "제거된 행 수 :",
        len(merged_df) - len(result_df)
    )

    return result_df

In [ ]:
# ================================================================================
# 10. 제품분석 통합
# ================================================================================

product_df = merge_sheet(
    sheet_name="제품분석",
    file_info=file_info,
    need_keyword=True
)


# ================================================================================
# 11. 가격요약 통합
# ================================================================================

price_summary_df = merge_sheet(
    sheet_name="가격요약",
    file_info=file_info,
    need_keyword=False
)


# ================================================================================
# 12. 제품분석 검색키워드 분포
# ================================================================================

print()
print("=" * 80)
print("⑧ 제품분석 최종 검색키워드 분포")
print("=" * 80)

if "검색키워드" in product_df.columns:

    keyword_counts = (
        product_df["검색키워드"]
        .fillna("")
        .astype(str)
        .str.strip()
        .value_counts()
    )

    print(keyword_counts)


# ================================================================================
# 13. Excel 내보내기
# ================================================================================

print()
print("=" * 80)
print("⑨ Excel 내보내기")
print("=" * 80)

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    # 통합 제품분석
    product_df.to_excel(
        writer,
        sheet_name="제품분석",
        index=False
    )

    # 통합 가격요약
    price_summary_df.to_excel(
        writer,
        sheet_name="가격요약",
        index=False
    )


print()
print("엑셀 저장 완료")
print()
print("저장 위치 :")
print(OUTPUT_FILE)

print()
print("파일명 :")
print(OUTPUT_FILE.name)

print()
print("제품분석 최종 행 수 :", len(product_df))
print("가격요약 최종 행 수 :", len(price_summary_df))


# ================================================================================
# 14. PC로 엑셀 다운로드
# ================================================================================

print()
print("=" * 80)
print("⑩ 엑셀 다운로드")
print("=" * 80)

files.download(
    str(OUTPUT_FILE)
)

print("다운로드 요청 완료")


# ================================================================================
# 15. 최종 확인
# ================================================================================

print()
print("=" * 80)
print("⑪ 최종 결과")
print("=" * 80)

print()
print("[제품분석]")
print(
    "행 수   :",
    len(product_df)
)

print(
    "컬럼 수 :",
    len(product_df.columns)
)

print()
print("[가격요약]")
print(
    "행 수   :",
    len(price_summary_df)
)

print(
    "컬럼 수 :",
    len(price_summary_df.columns)
)

print()
print("[제품분석 첫 5개]")
display(
    product_df.head()
)

print()
print("[가격요약 첫 5개]")
display(
    price_summary_df.head()
)

**가격변화 데이터 파일 append**



In [ ]:
# ================================================================================
# 쿠팡 가격이력만 별도 통합
#   - 제품분석 / 가격요약은 기존 코드 그대로 사용
#   - 가격이력은 상품 단위로 중복 제거하지 않음
#   - 같은 상품 + 같은 날짜/시간만 중복 제거
#   - 서로 다른 날짜/시간의 가격이력은 모두 유지
#   - 검색어 우선순위:
#       주름패치 > 피부패치 > 패치
#       성인용기저귀 > 기저귀
#       입는생리대 > 생리대
#       물티슈
# ================================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import files

# ================================================================================
# 1. Google Drive 폴더
# ================================================================================

INPUT_DIR = Path("/content/drive/MyDrive/YOUR_PROJECT_FOLDER/coupang_data/coupang_pricedata")

# ================================================================================
# 2. 검색어 우선순위
#    앞에 있을수록 먼저 유지
# ================================================================================

APPEND_ORDER = {
    "주름패치": 1,
    "피부패치": 2,
    "패치": 3,

    "성인용기저귀": 4,
    "기저귀": 5,

    "입는생리대": 6,
    "생리대": 7,

    "물티슈": 8,
}

# ================================================================================
# 3. ID 정규화
# ================================================================================

def normalize_id(x):
    if pd.isna(x):
        return ""

    s = str(x).strip()

    # 엑셀에서 123456789.0 형태로 들어온 경우
    if s.endswith(".0"):
        s = s[:-2]

    return s


# ================================================================================
# 4. 파일별 검색키워드 확인
#    우선 가격이력의 검색키워드
#    없으면 제품분석의 검색키워드 사용
# ================================================================================

def get_file_keyword(file_path):

    # ----------------------------
    # 가격이력에서 확인
    # ----------------------------
    try:
        df_hist = pd.read_excel(
            file_path,
            sheet_name="가격이력"
        )

        if "검색키워드" in df_hist.columns:

            keywords = (
                df_hist["검색키워드"]
                .dropna()
                .astype(str)
                .str.strip()
            )

            keywords = keywords[keywords != ""]

            if len(keywords) > 0:
                return keywords.iloc[0]

    except Exception:
        pass

    # ----------------------------
    # 제품분석에서 확인
    # ----------------------------
    try:
        df_product = pd.read_excel(
            file_path,
            sheet_name="제품분석"
        )

        if "검색키워드" in df_product.columns:

            keywords = (
                df_product["검색키워드"]
                .dropna()
                .astype(str)
                .str.strip()
            )

            keywords = keywords[keywords != ""]

            if len(keywords) > 0:
                return keywords.iloc[0]

    except Exception:
        pass

    return "검색어없음"


# ================================================================================
# 5. 가격이력 파일 목록 읽기
# ================================================================================

xlsx_files = sorted(
    [
        p for p in INPUT_DIR.glob("*.xlsx")
        if not p.name.startswith("~$")
        and "가격이력_통합" not in p.name
    ]
)

print("=" * 80)
print("가격이력 통합 대상 파일")
print("=" * 80)

file_info = []

for file_path in xlsx_files:

    keyword = get_file_keyword(file_path)

    append_order = APPEND_ORDER.get(
        keyword,
        999
    )

    file_info.append({
        "path": file_path,
        "keyword": keyword,
        "append_order": append_order
    })

    print(
        f"{file_path.name}  |  "
        f"검색키워드 = {keyword}  |  "
        f"우선순위 = {append_order}"
    )


# ================================================================================
# 6. 검색어 우선순위대로 정렬
# ================================================================================

file_info = sorted(
    file_info,
    key=lambda x: (
        x["append_order"],
        x["path"].name
    )
)

print("\n" + "=" * 80)
print("실제 append 순서")
print("=" * 80)

for i, info in enumerate(file_info, start=1):

    print(
        f"{i:02d}. "
        f"{info['keyword']}  ←  {info['path'].name}"
    )


# ================================================================================
# 7. 가격이력 읽어서 전부 append
# ================================================================================

all_history = []

for info in file_info:

    file_path = info["path"]
    keyword = info["keyword"]
    append_order = info["append_order"]

    try:
        df = pd.read_excel(
            file_path,
            sheet_name="가격이력"
        )

    except Exception as e:

        print(
            f"\n[읽기 실패] {file_path.name}"
        )
        print(e)

        continue

    if df.empty:
        continue

    # ------------------------------------------------------------
    # 검색키워드가 없거나 공백인 행은 파일 키워드로 채움
    # ------------------------------------------------------------

    if "검색키워드" not in df.columns:

        df.insert(
            1,
            "검색키워드",
            keyword
        )

    else:

        df["검색키워드"] = (
            df["검색키워드"]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        df.loc[
            df["검색키워드"] == "",
            "검색키워드"
        ] = keyword

    # ------------------------------------------------------------
    # 원본파일 / 우선순위 기록
    # ------------------------------------------------------------

    df["_원본파일"] = file_path.name
    df["_append순서"] = append_order

    all_history.append(df)


# ================================================================================
# 8. 전체 가격이력 합치기
# ================================================================================

if not all_history:

    raise ValueError(
        "가격이력 데이터를 읽지 못했습니다."
    )

price_history_df = pd.concat(
    all_history,
    ignore_index=True,
    sort=False
)

print("\n" + "=" * 80)
print("가격이력 전체 append 완료")
print("=" * 80)

print(
    f"전체 행 수 : {len(price_history_df):,}"
)


# ================================================================================
# 9. ID 컬럼 확인
# ================================================================================

for col in ["productId", "itemId", "vendorItemId"]:

    if col not in price_history_df.columns:
        price_history_df[col] = ""

    price_history_df[col] = (
        price_history_df[col]
        .apply(normalize_id)
    )


# ================================================================================
# 10. 날짜/시간 정규화
#     ★ 날짜만 자르는 것이 아니라 timestamp까지 유지
#     예:
#       2026-04-06 10:00
#       2026-04-06 16:00
#     → 서로 다른 이력으로 유지
# ================================================================================

if "날짜" not in price_history_df.columns:

    raise ValueError(
        "가격이력 시트에 '날짜' 컬럼이 없습니다."
    )

price_history_df["_날짜_key"] = pd.to_datetime(
    price_history_df["날짜"],
    errors="coerce"
)

# 날짜 파싱 결과 확인
invalid_date_count = price_history_df["_날짜_key"].isna().sum()

print(
    f"날짜 파싱 실패 : {invalid_date_count:,}행"
)


# ================================================================================
# 11. 가격이력 중복 제거
#
# ★ 핵심
#
# 제품분석:
#   상품 자체가 중복이면 제거
#
# 가격이력:
#   상품 + 동일 timestamp가 중복일 때만 제거
#
# 예:
#
# 피부패치 | 상품 A | 2026-04-06 10:00 | 5,460   ← 유지
# 피부패치 | 상품 A | 2026-04-06 16:00 | 5,800   ← 유지
# 패치     | 상품 A | 2026-04-06 10:00 | 5,460   ← 제거
# 패치     | 상품 A | 2026-04-06 16:00 | 5,800   ← 제거
# 패치     | 상품 A | 2026-04-07 10:00 | 5,900   ← 유지
#
# → 서로 다른 날짜/시간은 모두 유지
# ================================================================================

seen_item_date = set()
seen_vendor_date = set()

keep_mask = []

duplicate_count = 0

for _, row in price_history_df.iterrows():

    dt = row["_날짜_key"]

    item_id = row["itemId"]
    vendor_id = row["vendorItemId"]

    is_duplicate = False

    # ------------------------------------------------------------
    # 날짜가 정상적으로 파싱된 경우만
    # 상품 + 날짜/시간으로 중복 검사
    # ------------------------------------------------------------

    if pd.notna(dt):

        # itemId 기준
        if item_id != "":

            item_key = (
                item_id,
                dt
            )

            if item_key in seen_item_date:
                is_duplicate = True
            else:
                seen_item_date.add(item_key)

        # vendorItemId 기준
        if vendor_id != "":

            vendor_key = (
                vendor_id,
                dt
            )

            if vendor_key in seen_vendor_date:
                is_duplicate = True
            else:
                seen_vendor_date.add(vendor_key)

    # ------------------------------------------------------------
    # 같은 상품 + timestamp가 이미 존재하면 제거
    # ------------------------------------------------------------

    if is_duplicate:
        keep_mask.append(False)
        duplicate_count += 1
    else:
        keep_mask.append(True)


price_history_df = price_history_df.loc[
    keep_mask
].copy()


print("\n" + "=" * 80)
print("가격이력 중복 제거 완료")
print("=" * 80)

print(
    f"중복 제거 행 수 : {duplicate_count:,}"
)

print(
    f"최종 행 수      : {len(price_history_df):,}"
)


# ================================================================================
# 12. 날짜순 정렬
#
# 같은 timestamp에서 어떤 검색어의 행을 남길지는
# 이미 APPEND_ORDER로 결정됨
#
# 최종 출력은 시간순으로 보기 좋게 정렬
# ================================================================================

price_history_df = price_history_df.sort_values(
    by=[
        "_날짜_key",
        "_append순서"
    ],
    ascending=[
        True,
        True
    ],
    na_position="last"
).reset_index(drop=True)


# ================================================================================
# 13. 임시 컬럼 삭제
# ================================================================================

price_history_df = price_history_df.drop(
    columns=[
        "_날짜_key",
        "_append순서"
    ],
    errors="ignore"
)


# ================================================================================
# 14. 원본파일 컬럼은 맨 뒤로
# ================================================================================

if "_원본파일" in price_history_df.columns:

    cols = [
        c
        for c in price_history_df.columns
        if c != "_원본파일"
    ]

    price_history_df = price_history_df[
        cols + ["_원본파일"]
    ]


# ================================================================================
# 15. 결과 확인
# ================================================================================

print("\n" + "=" * 80)
print("최종 가격이력")
print("=" * 80)

print(
    f"최종 행 수 : {len(price_history_df):,}"
)

print(
    f"최종 열 수 : {len(price_history_df.columns):,}"
)

display(
    price_history_df.head(20)
)


# ================================================================================
# 16. 별도 Excel 저장
# ================================================================================

OUTPUT_FILE = (
    INPUT_DIR /
    "쿠팡_전체_가격이력_통합.xlsx"
)

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    price_history_df.to_excel(
        writer,
        sheet_name="가격이력",
        index=False
    )


# ================================================================================
# 17. 저장 완료
# ================================================================================

print("\n" + "=" * 80)
print("가격이력 통합 파일 저장 완료")
print("=" * 80)

print(
    f"저장 위치 : {OUTPUT_FILE}"
)


# ================================================================================
# 18. PC로 다운로드
# ================================================================================

files.download(
    str(OUTPUT_FILE)
)